# MAJ-Debate: Stage 1 Side-Picking Agent Generation

This notebook runs real Stage 1 generation only. It loads topics from `data/eval/*.jsonl`, calls the live model, and writes split-specific outputs under `outputs/stage1/`.


## 0. Imports & Configuration


In [1]:
import os, json, time, logging
from pathlib import Path
from datetime import datetime
import requests
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists()), cwd)
ENV_PATH = PROJECT_ROOT / '.env'
load_dotenv(ENV_PATH if ENV_PATH.exists() else None)

OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY is required for Stage 1.')

# NOTE: OpenRouter slugs for Gemma 3 are '...-it' (instruction-tuned) or '...-it:free'.
# The previous default 'google/gemma-3-12b' is not a valid model id and returns 404.
OPENROUTER_MODEL = os.environ.get('MAJ_OPENROUTER_MODEL', 'google/gemma-3-12b-it')
OPENROUTER_URL = os.environ.get('MAJ_OPENROUTER_URL', 'https://openrouter.ai/api/v1/chat/completions')
OPENROUTER_SITE_URL = os.environ.get('OPENROUTER_SITE_URL', 'http://localhost')
OPENROUTER_APP_NAME = os.environ.get('OPENROUTER_APP_NAME', 'MAJ-Debate-Stage1')

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://YOUR_VM_IP:5000')
MODEL = OPENROUTER_MODEL
TEMPERATURE = 0.7
MAX_TOKENS = int(os.environ.get('MAJ_STAGE1_MAX_TOKENS', '1024'))
RATE_LIMIT_SLEEP = 1.2
MAX_RETRIES = 3

N_PRO = int(os.environ.get('MAJ_STAGE1_N_PRO', '3'))
N_CON = int(os.environ.get('MAJ_STAGE1_N_CON', '3'))
ARGS_PER_AGENT_R1 = int(os.environ.get('MAJ_STAGE1_R1_ARGS', '3'))
ARGS_PER_AGENT_R2 = int(os.environ.get('MAJ_STAGE1_R2_ARGS', '2'))

EVAL_SPLIT = os.environ.get('MAJ_EVAL_SPLIT', 'ddo_sample')
TOPIC_LIMIT = int(os.environ.get('MAJ_STAGE1_TOPIC_LIMIT', '0'))
TOPIC_FILE_ENV = os.environ.get('MAJ_TOPIC_FILE')

DEFAULT_TOPIC_FILE = PROJECT_ROOT / 'data' / 'eval' / f'{EVAL_SPLIT}_topics.jsonl'
if TOPIC_FILE_ENV:
    topic_path = Path(TOPIC_FILE_ENV)
    TOPIC_FILE = topic_path if topic_path.is_absolute() else (PROJECT_ROOT / topic_path)
else:
    TOPIC_FILE = DEFAULT_TOPIC_FILE
TOPIC_FILE = TOPIC_FILE.resolve()

OUT_DIR = PROJECT_ROOT / 'outputs' / 'stage1' / EVAL_SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'stage1_arguments.json'
FAILURES_FILE = OUT_DIR / 'stage1_failures.json'
PERSONA_FILE = OUT_DIR / 'personas.json'
RESUME_EXISTING = os.environ.get('MAJ_STAGE1_RESUME', '1').strip() not in {'0', 'false', 'False'}
CONTINUE_ON_ERROR = os.environ.get('MAJ_STAGE1_CONTINUE_ON_ERROR', '1').strip() not in {'0', 'false', 'False'}

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('stage1')

ACTIVE_MODEL = MODEL

print(f'Project root       : {PROJECT_ROOT}')
print(f'.env loaded from   : {ENV_PATH if ENV_PATH.exists() else "not found"}')
print('LLM provider       : openrouter')
print(f'OpenRouter URL     : {OPENROUTER_URL}')
print(f'OpenRouter model   : {MODEL}')
print(f'Active model       : {ACTIVE_MODEL}')
print(f'Max tokens         : {MAX_TOKENS}')
print(f'Eval split         : {EVAL_SPLIT}')
print(f'Topic file         : {TOPIC_FILE}')
print(f'Agents             : {N_PRO} Pro + {N_CON} Con')
print(f'Args/agent R1/R2   : {ARGS_PER_AGENT_R1} / {ARGS_PER_AGENT_R2}')
print(f'Output             : {OUT_FILE.resolve()}')
print(f'Failures file      : {FAILURES_FILE.resolve()}')
print(f'Resume existing    : {RESUME_EXISTING}')
print(f'Continue on error  : {CONTINUE_ON_ERROR}')


Project root       : C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate
.env loaded from   : C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\.env
LLM provider       : openrouter
OpenRouter URL     : https://openrouter.ai/api/v1/chat/completions
OpenRouter model   : google/gemma-3-12b-it
Active model       : google/gemma-3-12b-it
Max tokens         : 1024
Eval split         : ddo_sample
Topic file         : C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\data\eval\ddo_sample_topics.jsonl
Agents             : 3 Pro + 3 Con
Args/agent R1/R2   : 3 / 2
Output             : C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\outputs\stage1\ddo_sample\stage1_arguments.json
Failures file      : C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\outputs\stage1\ddo_sample\stage1_failures.json
Resume existing    : True
Continue on error  : True


## 1. Personas and Topic Loading


In [2]:
PRO_PERSONAS = [
    {'id': 'pro_rationalist', 'name': 'Rationalist Pro', 'stance': 'PRO', 'reasoning_style': 'logical-empirical', 'rhetorical_mode': 'cite quantitative evidence and causal mechanisms', 'description': 'Argues from data, statistics, and formal logic. Prioritises measurable outcomes.'},
    {'id': 'pro_ethicist', 'name': 'Ethics Advocate Pro', 'stance': 'PRO', 'reasoning_style': 'ethical-normative', 'rhetorical_mode': 'appeal to moral principles and rights-based frameworks', 'description': 'Argues from fairness and justice. References established ethical frameworks.'},
    {'id': 'pro_futurist', 'name': 'Futurist Pro', 'stance': 'PRO', 'reasoning_style': 'economic-consequentialist', 'rhetorical_mode': 'project long-term societal and economic benefits', 'description': 'Argues from systemic benefits and long-horizon impact. Accepts short-term trade-offs.'},
]
CON_PERSONAS = [
    {'id': 'con_skeptic', 'name': 'Skeptic Con', 'stance': 'CON', 'reasoning_style': 'logical-empirical', 'rhetorical_mode': 'challenge evidence quality and burden of proof', 'description': 'Contests factual claims, demands rigorous evidence, identifies logical fallacies.'},
    {'id': 'con_rights', 'name': 'Rights Defender Con', 'stance': 'CON', 'reasoning_style': 'ethical-normative', 'rhetorical_mode': 'appeal to human rights, procedural justice, and democratic accountability', 'description': 'Argues the proposal violates fundamental rights regardless of practical merits.'},
    {'id': 'con_pragmatist', 'name': 'Pragmatist Con', 'stance': 'CON', 'reasoning_style': 'economic-consequentialist', 'rhetorical_mode': 'highlight implementation barriers and unintended consequences', 'description': 'Argues from practical constraints: cost, feasibility, second-order effects.'},
]
ACTIVE_PRO = PRO_PERSONAS[:N_PRO]
ACTIVE_CON = CON_PERSONAS[:N_CON]

def normalize_topic(raw, idx):
    topic_id = raw.get('topic_id') or raw.get('id') or f'{EVAL_SPLIT.upper()}_{idx:04d}'
    topic_text = raw.get('topic_text') or raw.get('text')
    if not topic_text:
        raise ValueError(f'Missing topic_text/text for record {topic_id}')
    return {
        'topic_id': topic_id,
        'topic_text': topic_text,
        'domain': raw.get('domain', 'unknown'),
        'benchmark_label': raw.get('benchmark_label'),
        'source_dataset': raw.get('source_dataset', EVAL_SPLIT),
        'source_ref': raw.get('source_ref'),
    }

def load_topics(path):
    if not path.exists():
        raise FileNotFoundError(f'Topic file not found: {path}. Create a real dataset file under data/eval/.')
    topics = []
    if path.suffix.lower() == '.jsonl':
        with open(path, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f, start=1):
                line = line.strip()
                if line:
                    topics.append(normalize_topic(json.loads(line), idx))
    else:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        rows = data['topics'] if isinstance(data, dict) and 'topics' in data else data
        topics = [normalize_topic(raw, idx) for idx, raw in enumerate(rows, start=1)]
    if TOPIC_LIMIT > 0:
        topics = topics[:TOPIC_LIMIT]
    if not topics:
        raise ValueError('No topics loaded. Provide a non-empty real topic file.')
    return topics

TOPICS = load_topics(TOPIC_FILE)
print(f'Total topics loaded: {len(TOPICS)}')
print(json.dumps(TOPICS[0], indent=2))


Total topics loaded: 500
{
  "topic_id": "DDO_00615",
  "topic_text": "A free market devoid of all government intervention would hurt the U.S. economy",
  "domain": "Economics",
  "benchmark_label": "CON",
  "source_dataset": "DDO",
  "source_ref": "A-free-market-devoid-of-all-government-intervention-would-hurt-the-U.S.-economy/2/"
}


## 2. Prompt Templates and API Client


In [3]:
import re


def build_r1_prompt(topic, persona, n_args):
    return (
        f'You are a debate agent with the following profile:\n'
        f'- Name            : {persona["name"]}\n'
        f'- Stance          : {persona["stance"]} (argue FOR this position only)\n'
        f'- Reasoning style : {persona["reasoning_style"]}\n'
        f'- Rhetorical mode : {persona["rhetorical_mode"]}\n'
        f'- Profile         : {persona["description"]}\n\n'
        f'Debate topic: "{topic}"\n\n'
        f'Generate exactly {n_args} distinct high-quality arguments for the {persona["stance"]} position.\n'
        f'Each argument must be a single clear sentence, max 40 words, and substantively different.\n\n'
        f'Output ONLY a valid JSON array of strings.\n'
        f'["argument 1", "argument 2", "argument 3"]'
    )


def build_r2_prompt(topic, persona, opposing_args, n_args):
    numbered = '\n'.join(f'  [{i+1}] {a}' for i, a in enumerate(opposing_args))
    return (
        f'You are a debate agent with the following profile:\n'
        f'- Name            : {persona["name"]}\n'
        f'- Stance          : {persona["stance"]}\n'
        f'- Reasoning style : {persona["reasoning_style"]}\n'
        f'- Rhetorical mode : {persona["rhetorical_mode"]}\n\n'
        f'Debate topic: "{topic}"\n\n'
        f'The opposing side has made these arguments (read-only context):\n{numbered}\n\n'
        f'Generate exactly {n_args} targeted counter-arguments from your {persona["stance"]} position.\n'
        f'Each counter-argument must directly attack one specific opposing argument, be a single clear sentence, and be at most 30 words.\n\n'
        f'Output ONLY a valid JSON array of objects.\n'
        '[{"targets_arg": 1, "argument": "..."}]'
    )


NO_THINKING_INSTRUCTION = (
    'Do not think step by step. Do not include chain-of-thought, analysis, or commentary. '
    'Return only the requested JSON and nothing else.'
)


def _build_payload(prompt, max_tokens):
    """Build the OpenRouter chat completion payload.

    We explicitly disable reasoning/thinking tokens via the OpenRouter
    `reasoning.exclude` flag. Without this, some models (Gemma 3, Qwen,
    DeepSeek, o-series) can burn the entire max_tokens budget on hidden
    reasoning and return empty `content`, which was causing the
    "OpenRouter returned empty content" RuntimeError in earlier runs.
    """
    return {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': NO_THINKING_INSTRUCTION},
            {'role': 'user', 'content': prompt},
        ],
        'temperature': TEMPERATURE,
        'max_tokens': max_tokens,
        'reasoning': {'exclude': True, 'enabled': False},
    }


def call_openrouter(prompt):
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
        'HTTP-Referer': OPENROUTER_SITE_URL,
        'X-Title': OPENROUTER_APP_NAME,
    }

    current_max_tokens = MAX_TOKENS
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        payload = _build_payload(prompt, current_max_tokens)

        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=120)
        except requests.RequestException as ex:
            last_error = ex
            log.warning('OpenRouter network error (attempt %s/%s): %s', attempt, MAX_RETRIES, ex)
            time.sleep(min(2 ** attempt, 30))
            continue

        # Rate limiting - respect Retry-After if the server sent one.
        if resp.status_code == 429:
            retry_after = resp.headers.get('Retry-After')
            try:
                wait_s = float(retry_after) if retry_after else 5 * attempt
            except ValueError:
                wait_s = 5 * attempt
            last_error = RuntimeError(f'OpenRouter 429 rate limit (retry after {wait_s}s): {resp.text[:200]}')
            log.warning('OpenRouter rate-limited (attempt %s/%s); sleeping %.1fs', attempt, MAX_RETRIES, wait_s)
            time.sleep(wait_s)
            continue

        # Transient server errors - back off and retry.
        if resp.status_code in (500, 502, 503, 504):
            last_error = RuntimeError(f'OpenRouter {resp.status_code}: {resp.text[:200]}')
            log.warning('OpenRouter %s (attempt %s/%s); backing off', resp.status_code, attempt, MAX_RETRIES)
            time.sleep(min(2 ** attempt, 30))
            continue

        if resp.status_code != 200:
            # Non-retryable (400/401/402/403/404) - fail fast with context.
            raise RuntimeError(
                f'OpenRouter API error {resp.status_code}: {resp.text[:500]}. '
                f'Check model slug ({MODEL!r}), API key, and account credits.'
            )

        try:
            data = resp.json()
        except ValueError as ex:
            last_error = RuntimeError(f'OpenRouter returned non-JSON body: {resp.text[:300]}')
            time.sleep(RATE_LIMIT_SLEEP * attempt)
            continue

        # OpenRouter sometimes returns an error object with a 200 status.
        if isinstance(data, dict) and data.get('error'):
            err = data['error']
            err_msg = err.get('message', str(err)) if isinstance(err, dict) else str(err)
            raise RuntimeError(f'OpenRouter error payload: {err_msg}')

        choices = data.get('choices') or []
        if not choices:
            raise RuntimeError(f'OpenRouter returned no choices: {str(data)[:300]}')

        choice = choices[0]
        finish_reason = choice.get('finish_reason') or choice.get('native_finish_reason')
        message = choice.get('message', {}) or {}
        content = message.get('content', '')
        if isinstance(content, list):
            content = ''.join(part.get('text', '') for part in content if isinstance(part, dict))
        content = (content or '').strip()

        if content:
            return content

        # Empty content. Figure out why.
        usage = data.get('usage', {}) or {}
        reasoning_tokens = (
            usage.get('reasoning_tokens')
            or (usage.get('completion_tokens_details') or {}).get('reasoning_tokens')
        )

        if finish_reason == 'length':
            # Truncated before any visible content was emitted. Try once more
            # with a bigger budget rather than failing the run.
            if current_max_tokens < 4096:
                current_max_tokens = min(current_max_tokens * 2, 4096)
                log.warning(
                    'Empty content with finish_reason=length (reasoning_tokens=%s); '
                    'retrying with max_tokens=%s',
                    reasoning_tokens, current_max_tokens,
                )
                time.sleep(RATE_LIMIT_SLEEP)
                continue

        last_error = RuntimeError(
            f'OpenRouter returned empty content (finish_reason={finish_reason!r}, '
            f'reasoning_tokens={reasoning_tokens}, usage={usage})'
        )
        time.sleep(RATE_LIMIT_SLEEP * attempt)

    raise RuntimeError(f'OpenRouter call failed after {MAX_RETRIES} attempts: {last_error}')


def clean_llm_text(text):
    text = text.strip().replace('\u201c', '"').replace('\u201d', '"').replace('\u2019', "'")
    if '```' in text:
        blocks = text.split('```')
        if len(blocks) >= 3:
            text = blocks[1]
            if text.lstrip().lower().startswith('json'):
                text = text.lstrip()[4:]
            text = text.strip()
    return text


def _extract_first_json_array(text):
    start = text.find('[')
    if start < 0:
        return text

    depth = 0
    in_string = False
    escaped = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escaped:
                escaped = False
            elif ch == '\\':
                escaped = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == '[':
            depth += 1
        elif ch == ']':
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return text[start:]


def _repair_common_json_issues(text):
    # Remove trailing commas before closing braces/brackets.
    repaired = re.sub(r',\s*([}\]])', r'\1', text)
    # Add commas between adjacent JSON strings: "a" "b" -> "a", "b".
    repaired = re.sub(r'"\s+"', '", "', repaired)
    # Add commas between adjacent objects in arrays: } { -> }, {.
    repaired = re.sub(r'}\s*{', '}, {', repaired)
    return repaired


def parse_json_list(text):
    cleaned = clean_llm_text(text)
    cleaned = _extract_first_json_array(cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as first_err:
        repaired = _repair_common_json_issues(cleaned)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError as second_err:
            preview = repaired[:300].replace('\n', ' ')
            raise ValueError(
                f'Failed to parse model JSON output. First error: {first_err}. '
                f'After repair: {second_err}. Preview: {preview}'
            ) from second_err


## 3. Core Stage 1 Functions


In [4]:
MAX_JSON_PARSE_RETRIES = int(os.environ.get('MAJ_STAGE1_PARSE_RETRIES', '3'))


def _parse_with_retry(prompt, stage_tag, topic_id, persona_name):
    last_error = None
    for attempt in range(1, MAX_JSON_PARSE_RETRIES + 1):
        raw = None
        try:
            raw = call_openrouter(prompt)
            parsed = parse_json_list(raw)
            if not isinstance(parsed, list) or len(parsed) == 0:
                raise ValueError(f'Expected non-empty JSON list, got {type(parsed)} with size {len(parsed) if isinstance(parsed, list) else "n/a"}')
            return parsed
        except RuntimeError as ex:
            last_error = ex
            log.warning('%s call failed | %s | %s | attempt %s/%s | %s', stage_tag, topic_id, persona_name, attempt, MAX_JSON_PARSE_RETRIES, ex)
            time.sleep(1.2 * attempt)
            continue
        except Exception as ex:
            last_error = ex
            log.warning('%s parse failed | %s | %s | attempt %s/%s | %s', stage_tag, topic_id, persona_name, attempt, MAX_JSON_PARSE_RETRIES, ex)

            # Ask the model to minimally rewrite its own output into valid JSON.
            if raw:
                fixer_prompt = (
                    'Rewrite the following content into strictly valid JSON only. '
                    'Do not add commentary or markdown fences. Preserve the same items and meaning.\n\n'
                    f'CONTENT:\n{raw}'
                )
                try:
                    fixed_raw = call_openrouter(fixer_prompt)
                    parsed_fixed = parse_json_list(fixed_raw)
                    if isinstance(parsed_fixed, list) and len(parsed_fixed) > 0:
                        return parsed_fixed
                    raise ValueError('Fixer returned empty or non-list JSON output')
                except Exception as fix_ex:
                    last_error = fix_ex
                    log.warning('%s fixer failed | %s | %s | attempt %s/%s | %s', stage_tag, topic_id, persona_name, attempt, MAX_JSON_PARSE_RETRIES, fix_ex)

            time.sleep(0.8 * attempt)

    raise ValueError(f'{stage_tag} JSON parsing failed for {persona_name} on {topic_id} after {MAX_JSON_PARSE_RETRIES} attempts: {last_error}')


def run_subround1(topic, personas, n_args):
    results = {}
    for persona in personas:
        log.info('R1 | %s | %s', topic['topic_id'], persona['name'])
        prompt = build_r1_prompt(topic['topic_text'], persona, n_args)

        args = []
        for attempt in range(1, MAX_JSON_PARSE_RETRIES + 1):
            parsed = _parse_with_retry(prompt, 'R1', topic['topic_id'], persona['name'])
            args = [str(a).strip() for a in parsed if str(a).strip()][:n_args]
            if len(args) == n_args:
                break
            log.warning('R1 shape failed | %s | %s | attempt %s/%s | expected %s args, got %s', topic['topic_id'], persona['name'], attempt, MAX_JSON_PARSE_RETRIES, n_args, len(args))
            time.sleep(0.5 * attempt)

        if len(args) != n_args:
            raise ValueError(f"Expected {n_args} R1 args, got {len(args)} for {persona['name']} on {topic['topic_id']}")

        results[persona['id']] = {'persona': persona, 'arguments': args, 'round': 1}
        time.sleep(RATE_LIMIT_SLEEP)
    return results


def run_subround2(topic, personas, r1_opposing, n_args):
    opposing_args = []
    for v in r1_opposing.values():
        opposing_args.extend(v['arguments'])
    results = {}
    for persona in personas:
        log.info('R2 | %s | %s', topic['topic_id'], persona['name'])
        prompt = build_r2_prompt(topic['topic_text'], persona, opposing_args, n_args)

        counter_args = []
        for attempt in range(1, MAX_JSON_PARSE_RETRIES + 1):
            parsed = _parse_with_retry(prompt, 'R2', topic['topic_id'], persona['name'])
            counter_args = []
            for item in parsed[:n_args]:
                if not isinstance(item, dict):
                    counter_args = []
                    break
                counter_args.append({'targets_arg': item.get('targets_arg', -1), 'argument': str(item.get('argument', '')).strip()})

            if len(counter_args) == n_args:
                break

            log.warning('R2 shape failed | %s | %s | attempt %s/%s | expected %s args, got %s', topic['topic_id'], persona['name'], attempt, MAX_JSON_PARSE_RETRIES, n_args, len(counter_args))
            time.sleep(0.5 * attempt)

        if len(counter_args) != n_args:
            raise ValueError(f"Expected {n_args} R2 args, got {len(counter_args)} for {persona['name']} on {topic['topic_id']}")

        results[persona['id']] = {'persona': persona, 'counter_args': counter_args, 'opposing_context': opposing_args, 'round': 2}
        time.sleep(RATE_LIMIT_SLEEP)
    return results


try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('MAJ-Debate')
    MLFLOW_OK = True
except Exception as ex:
    MLFLOW_OK = False
    print(f'MLflow unavailable ({ex}) - results saved locally only')


def mlflow_log(run_name, params, metrics, artifact_paths):
    if not MLFLOW_OK:
        return
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for path in artifact_paths:
            mlflow.log_artifact(str(path), artifact_path=f'stage1/{EVAL_SPLIT}')


MLflow unavailable (No module named 'pkg_resources') - results saved locally only


## 4. Main Run - Stage 1 on All Topics


In [5]:
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'stage1-{EVAL_SPLIT}-{RUN_TS}'

def build_topic_result(topic, pro_r1, con_r1, pro_r2, con_r2):
    r1_pro = sum(len(v['arguments']) for v in pro_r1.values())
    r1_con = sum(len(v['arguments']) for v in con_r1.values())
    r2_pro = sum(len(v['counter_args']) for v in pro_r2.values())
    r2_con = sum(len(v['counter_args']) for v in con_r2.values())
    flat_args = []
    idx = 0
    for pid, d in pro_r1.items():
        for arg in d['arguments']:
            flat_args.append({'arg_id': f"{topic['topic_id']}_A{idx:03d}", 'persona_id': pid, 'persona': d['persona']['name'], 'stance': 'PRO', 'round': 1, 'targets_arg': None, 'text': arg})
            idx += 1
    for pid, d in con_r1.items():
        for arg in d['arguments']:
            flat_args.append({'arg_id': f"{topic['topic_id']}_A{idx:03d}", 'persona_id': pid, 'persona': d['persona']['name'], 'stance': 'CON', 'round': 1, 'targets_arg': None, 'text': arg})
            idx += 1
    for pid, d in pro_r2.items():
        for ca in d['counter_args']:
            flat_args.append({'arg_id': f"{topic['topic_id']}_A{idx:03d}", 'persona_id': pid, 'persona': d['persona']['name'], 'stance': 'PRO', 'round': 2, 'targets_arg': ca.get('targets_arg'), 'text': ca['argument']})
            idx += 1
    for pid, d in con_r2.items():
        for ca in d['counter_args']:
            flat_args.append({'arg_id': f"{topic['topic_id']}_A{idx:03d}", 'persona_id': pid, 'persona': d['persona']['name'], 'stance': 'CON', 'round': 2, 'targets_arg': ca.get('targets_arg'), 'text': ca['argument']})
            idx += 1
    return {
        'topic_id': topic['topic_id'],
        'topic_text': topic['topic_text'],
        'domain': topic['domain'],
        'benchmark_label': topic['benchmark_label'],
        'source_dataset': topic['source_dataset'],
        'source_ref': topic['source_ref'],
        'evaluation_split': EVAL_SPLIT,
        'run_name': RUN_NAME,
        'arguments': flat_args,
        'meta': {'n_pro': N_PRO, 'n_con': N_CON, 'r1_per_agent': ARGS_PER_AGENT_R1, 'r2_per_agent': ARGS_PER_AGENT_R2, 'total_args': len(flat_args), 'r1_args': r1_pro + r1_con, 'r2_args': r2_pro + r2_con, 'model': ACTIVE_MODEL, 'provider': 'openrouter'},
    }

def compute_summary(topics_payload):
    total_r1_args = sum(t.get('meta', {}).get('r1_args', 0) for t in topics_payload)
    total_r2_args = sum(t.get('meta', {}).get('r2_args', 0) for t in topics_payload)
    total_args = total_r1_args + total_r2_args
    total_topics = len(topics_payload)
    return {
        'total_topics': total_topics,
        'total_r1_args': total_r1_args,
        'total_r2_args': total_r2_args,
        'total_args': total_args,
        'avg_args_per_topic': round(total_args / total_topics, 1) if total_topics else 0.0,
        'r2_coverage_pct': round(total_r2_args / max(total_args, 1) * 100, 1),
    }

def save_stage1_state(topic_results_by_id, failures):
    ordered_topics = [topic_results_by_id[t['topic_id']] for t in TOPICS if t['topic_id'] in topic_results_by_id]
    output_doc = {
        'stage': 1,
        'run_name': RUN_NAME,
        'timestamp': RUN_TS,
        'config': {'provider': 'openrouter', 'model': ACTIVE_MODEL, 'openrouter_url': OPENROUTER_URL, 'n_pro': N_PRO, 'n_con': N_CON, 'r1_per_agent': ARGS_PER_AGENT_R1, 'r2_per_agent': ARGS_PER_AGENT_R2, 'evaluation_split': EVAL_SPLIT, 'topic_file': str(TOPIC_FILE), 'resume_existing': RESUME_EXISTING},
        'personas': {'pro': ACTIVE_PRO, 'con': ACTIVE_CON},
        'topics': ordered_topics,
        'summary': compute_summary(ordered_topics),
    }
    with open(OUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(output_doc, f, indent=2)
    with open(PERSONA_FILE, 'w', encoding='utf-8') as f:
        json.dump({'pro': ACTIVE_PRO, 'con': ACTIVE_CON}, f, indent=2)
    with open(FAILURES_FILE, 'w', encoding='utf-8') as f:
        json.dump({'run_name': RUN_NAME, 'timestamp': RUN_TS, 'failed_topics': failures}, f, indent=2)
    return output_doc

existing_topics_by_id = {}
if RESUME_EXISTING and OUT_FILE.exists():
    try:
        with open(OUT_FILE, 'r', encoding='utf-8') as f:
            existing_doc = json.load(f)
        for topic_payload in existing_doc.get('topics', []):
            topic_id = topic_payload.get('topic_id')
            if topic_id and topic_payload.get('arguments'):
                existing_topics_by_id[topic_id] = topic_payload
        print(f'Resuming from existing output: {len(existing_topics_by_id)} completed topics found')
    except Exception as ex:
        print(f'Could not resume from existing output ({ex}); starting fresh')
        existing_topics_by_id = {}

existing_failures_by_id = {}
if FAILURES_FILE.exists():
    try:
        with open(FAILURES_FILE, 'r', encoding='utf-8') as f:
            existing_failures_doc = json.load(f)
        for failure_payload in existing_failures_doc.get('failed_topics', []):
            failed_topic_id = failure_payload.get('topic_id')
            if failed_topic_id and failed_topic_id not in existing_topics_by_id:
                existing_failures_by_id[failed_topic_id] = failure_payload
        if existing_failures_by_id:
            print(f'Retrying unresolved failures from prior runs: {len(existing_failures_by_id)} topics')
    except Exception as ex:
        print(f'Could not load existing failures ({ex}); continuing with a fresh failure log')
        existing_failures_by_id = {}

topic_results_by_id = dict(existing_topics_by_id)
failures_by_id = dict(existing_failures_by_id)

for i, topic in enumerate(TOPICS, start=1):
    topic_id = topic['topic_id']
    if topic_id in topic_results_by_id:
        print(f"[{i}/{len(TOPICS)}] {topic_id}: already completed, skipping")
        continue
    print(f"[{i}/{len(TOPICS)}] {topic_id}: {topic['topic_text']}")
    try:
        pro_r1 = run_subround1(topic, ACTIVE_PRO, ARGS_PER_AGENT_R1)
        con_r1 = run_subround1(topic, ACTIVE_CON, ARGS_PER_AGENT_R1)
        pro_r2 = run_subround2(topic, ACTIVE_PRO, r1_opposing=con_r1, n_args=ARGS_PER_AGENT_R2)
        con_r2 = run_subround2(topic, ACTIVE_CON, r1_opposing=pro_r1, n_args=ARGS_PER_AGENT_R2)
        topic_results_by_id[topic_id] = build_topic_result(topic, pro_r1, con_r1, pro_r2, con_r2)
        failures_by_id.pop(topic_id, None)
        output_doc = save_stage1_state(topic_results_by_id, list(failures_by_id.values()))
        print(f"  saved checkpoint after {topic_id}")
    except Exception as ex:
        log.exception('Stage 1 failed for %s', topic_id)
        failures_by_id[topic_id] = {'topic_id': topic_id, 'topic_text': topic['topic_text'], 'error': str(ex), 'run_name': RUN_NAME}
        output_doc = save_stage1_state(topic_results_by_id, list(failures_by_id.values()))
        if not CONTINUE_ON_ERROR:
            raise

output_doc = save_stage1_state(topic_results_by_id, list(failures_by_id.values()))
mlflow_log(run_name=RUN_NAME, params=output_doc['config'], metrics={k: float(v) for k, v in output_doc['summary'].items() if isinstance(v, (int, float))}, artifact_paths=[OUT_FILE, PERSONA_FILE, FAILURES_FILE])
print(f'Saved: {OUT_FILE.resolve()}')
print(output_doc['summary'])
print(f'Failed topics: {len(failures_by_id)}')


2026-04-17 23:56:09,837 INFO R1 | DDO_22091 | Rationalist Pro


Resuming from existing output: 117 completed topics found
Retrying unresolved failures from prior runs: 19 topics
[1/500] DDO_00615: already completed, skipping
[2/500] DDO_00889: already completed, skipping
[3/500] DDO_00700: already completed, skipping
[4/500] DDO_00714: already completed, skipping
[5/500] DDO_00437: already completed, skipping
[6/500] DDO_00941: already completed, skipping
[7/500] DDO_00662: already completed, skipping
[8/500] DDO_00928: already completed, skipping
[9/500] DDO_00660: already completed, skipping
[10/500] DDO_03153: already completed, skipping
[11/500] DDO_00701: already completed, skipping
[12/500] DDO_00716: already completed, skipping
[13/500] DDO_00519: already completed, skipping
[14/500] DDO_04696: already completed, skipping
[15/500] DDO_04372: already completed, skipping
[16/500] DDO_05888: already completed, skipping
[17/500] DDO_00710: already completed, skipping
[18/500] DDO_03223: already completed, skipping
[19/500] DDO_01034: already com

2026-04-17 23:56:13,426 INFO R1 | DDO_22091 | Ethics Advocate Pro
2026-04-17 23:56:17,722 INFO R1 | DDO_22091 | Futurist Pro
2026-04-17 23:56:22,461 INFO R1 | DDO_22091 | Skeptic Con
2026-04-17 23:56:29,099 INFO R1 | DDO_22091 | Rights Defender Con
2026-04-17 23:56:33,316 INFO R1 | DDO_22091 | Pragmatist Con
2026-04-17 23:56:36,901 INFO R2 | DDO_22091 | Rationalist Pro
2026-04-17 23:56:40,915 INFO R2 | DDO_22091 | Ethics Advocate Pro
2026-04-17 23:56:45,214 INFO R2 | DDO_22091 | Futurist Pro
2026-04-17 23:56:49,467 INFO R2 | DDO_22091 | Skeptic Con
2026-04-17 23:56:53,466 INFO R2 | DDO_22091 | Rights Defender Con
2026-04-17 23:56:58,016 INFO R2 | DDO_22091 | Pragmatist Con
2026-04-17 23:57:02,451 INFO R1 | DDO_17527 | Rationalist Pro


  saved checkpoint after DDO_22091
[119/500] DDO_17527: Education should be given more priority than security in the society


2026-04-17 23:57:06,802 INFO R1 | DDO_17527 | Ethics Advocate Pro
2026-04-17 23:57:11,875 INFO R1 | DDO_17527 | Futurist Pro
2026-04-17 23:57:15,981 INFO R1 | DDO_17527 | Skeptic Con
2026-04-17 23:57:20,435 INFO R1 | DDO_17527 | Rights Defender Con
2026-04-17 23:57:24,615 INFO R1 | DDO_17527 | Pragmatist Con
2026-04-17 23:57:28,079 INFO R2 | DDO_17527 | Rationalist Pro
2026-04-17 23:57:33,513 INFO R2 | DDO_17527 | Ethics Advocate Pro
2026-04-17 23:57:37,854 INFO R2 | DDO_17527 | Futurist Pro
2026-04-17 23:57:43,317 INFO R2 | DDO_17527 | Skeptic Con
2026-04-17 23:57:47,148 INFO R2 | DDO_17527 | Rights Defender Con
2026-04-17 23:57:52,364 INFO R2 | DDO_17527 | Pragmatist Con
2026-04-17 23:57:57,237 INFO R1 | DDO_52106 | Rationalist Pro


...

  saved checkpoint after DDO_20516
[499/500] DDO_72219: Universal Health Care Would be Beneficial to the U.S.


2026-04-18 05:48:52,901 INFO R1 | DDO_72219 | Ethics Advocate Pro
2026-04-18 05:48:56,223 INFO R1 | DDO_72219 | Futurist Pro
2026-04-18 05:49:01,109 INFO R1 | DDO_72219 | Skeptic Con
2026-04-18 05:49:04,621 INFO R1 | DDO_72219 | Rights Defender Con
2026-04-18 05:49:08,102 INFO R1 | DDO_72219 | Pragmatist Con
2026-04-18 05:49:14,758 INFO R2 | DDO_72219 | Rationalist Pro
2026-04-18 05:49:19,878 INFO R2 | DDO_72219 | Ethics Advocate Pro
2026-04-18 05:49:23,962 INFO R2 | DDO_72219 | Futurist Pro
2026-04-18 05:49:28,796 INFO R2 | DDO_72219 | Skeptic Con
2026-04-18 05:49:33,511 INFO R2 | DDO_72219 | Rights Defender Con
2026-04-18 05:49:38,617 INFO R2 | DDO_72219 | Pragmatist Con
2026-04-18 05:49:43,501 INFO R1 | DDO_03349 | Rationalist Pro


  saved checkpoint after DDO_72219
[500/500] DDO_03349: All Human Actions are Taken Through Self-Interest


2026-04-18 05:49:47,628 INFO R1 | DDO_03349 | Ethics Advocate Pro
2026-04-18 05:49:52,958 INFO R1 | DDO_03349 | Futurist Pro
2026-04-18 05:49:56,845 INFO R1 | DDO_03349 | Skeptic Con
2026-04-18 05:50:01,760 INFO R1 | DDO_03349 | Rights Defender Con
2026-04-18 05:50:06,665 INFO R1 | DDO_03349 | Pragmatist Con
2026-04-18 05:50:10,976 INFO R2 | DDO_03349 | Rationalist Pro
2026-04-18 05:50:16,813 INFO R2 | DDO_03349 | Ethics Advocate Pro
2026-04-18 05:50:21,350 INFO R2 | DDO_03349 | Futurist Pro
2026-04-18 05:50:24,960 INFO R2 | DDO_03349 | Skeptic Con
2026-04-18 05:50:29,613 INFO R2 | DDO_03349 | Rights Defender Con
2026-04-18 05:50:33,812 INFO R2 | DDO_03349 | Pragmatist Con


  saved checkpoint after DDO_03349
Saved: C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\outputs\stage1\ddo_sample\stage1_arguments.json
{'total_topics': 500, 'total_r1_args': 9000, 'total_r2_args': 6000, 'total_args': 15000, 'avg_args_per_topic': 30.0, 'r2_coverage_pct': 40.0}
Failed topics: 0


## 5. Inspect Output


In [6]:
sample = output_doc['topics'][0]
print('Topic :', sample['topic_text'])
print('Split :', sample['evaluation_split'])
print('Args  :', sample['meta']['total_args'])
print(json.dumps(sample['arguments'][0], indent=2))


Topic : A free market devoid of all government intervention would hurt the U.S. economy
Split : ddo_sample
Args  : 30
{
  "arg_id": "DDO_00615_A000",
  "persona_id": "pro_rationalist",
  "persona": "Rationalist Pro",
  "stance": "PRO",
  "round": 1,
  "targets_arg": null,
  "text": "Unregulated markets lead to monopolies, stifling innovation."
}


In [7]:
# Local parser tests (no API call) to validate malformed JSON handling.
_samples = [
    '["a" "b"]',
    '[{"targets_arg": 1, "argument": "x"} {"targets_arg": 2, "argument": "y"}]',
    '[{"targets_arg": 1, "argument": "x",}]',
]

for s in _samples:
    parsed = parse_json_list(s)
    print(type(parsed), len(parsed), parsed[0])

print('Local parser test passed')

<class 'list'> 2 a
<class 'list'> 2 {'targets_arg': 1, 'argument': 'x'}
<class 'list'> 1 {'targets_arg': 1, 'argument': 'x'}
Local parser test passed
